# 00 - Data prep: spatial split shared by Random Forest and U-Net

Run this once. It creates the **train / val / test split** that both models use, so their scores are comparable.

What it does:
1. Checks that all 16 TIFFs are readable, 7-band, and that the labeled area is the same in every year.
2. Cuts each tile into 1024 x 1024 pixel **blocks** and assigns whole blocks to train / val / test (about 70 / 15 / 15 % of labeled pixels). All years of one block go to the same split, so the same location never appears in both train and test.
3. Computes band mean/std from train blocks only (used by the U-Net).

Output files go to `WORK_DIR`: `splits.json`, `block_stats.csv`, `norm_stats.json`.

**Before running:** copy the 16 TIFFs to local disk (`TILE_DIR`) and check each file size matches the Drive listing.

In [1]:
!pip -q install rasterio

In [2]:
import os, re, json, glob, time
import numpy as np, pandas as pd
import rasterio
from rasterio.windows import Window

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

# ---------------- CONFIG (keep identical in all three notebooks) ----------------
TILE_DIR = "/content/tiles"                         # local copy of the 16 TIFFs (not the Drive mount)
WORK_DIR = "/content/drive/MyDrive/PreyLang_work"   # splits, stats, models and metrics are saved here
TILES    = ["0_0", "0_1", "1_0", "1_1"]
TRAIN_YEARS = [2019, 2021, 2023]                    # 2025 has no positive labels, so it is not used
BLOCK    = 1024      # spatial block size in pixels (about 10 km). A block never crosses a split.
PATCH    = 256       # U-Net patch size
SEED     = 42
BANDS    = ["B2", "B3", "B4", "B8", "B11", "B12"]
os.makedirs(WORK_DIR, exist_ok=True)
# --------------------------------------------------------------------------------

def tile_path(year, tile):
    return f"{TILE_DIR}/PreyLang_Composite_{year}_tile_{tile}.tif"

def read_window(year, tile, r0, c0, h, w):
    """Float32 array (7, h, w): B2 B3 B4 B8 B11 B12 label (NaN = missing)."""
    with rasterio.open(tile_path(year, tile)) as src:
        return src.read(window=Window(c0, r0, w, h)).astype("float32")

def add_indices(b):
    """b: (6, ...) ordered B2 B3 B4 B8 B11 B12 -> (3, ...) NDVI, NBR, NDWI."""
    eps = 1e-6
    ndvi = (b[3] - b[2]) / (b[3] + b[2] + eps)
    nbr  = (b[3] - b[5]) / (b[3] + b[5] + eps)
    ndwi = (b[1] - b[3]) / (b[1] + b[3] + eps)
    return np.stack([ndvi, nbr, ndwi])

Mounted at /content/drive


In [5]:
import shutil
import os

# Define where your TIFF files are located on Google Drive
# Update 'PreyLang_tiles' to the actual folder name on your Drive where the TIFFs reside
DRIVE_SOURCE_DIR = "/content/drive/MyDrive/PreyLang"

os.makedirs(TILE_DIR, exist_ok=True)

if os.path.exists(DRIVE_SOURCE_DIR):
    print(f"Copying TIFF files from {DRIVE_SOURCE_DIR} to {TILE_DIR}...")
    copied_count = 0
    for filename in os.listdir(DRIVE_SOURCE_DIR):
        if filename.endswith(".tif"):
            src_file = os.path.join(DRIVE_SOURCE_DIR, filename)
            dst_file = os.path.join(TILE_DIR, filename)
            shutil.copy(src_file, dst_file)
            copied_count += 1
    print(f"Successfully copied {copied_count} TIFF files to {TILE_DIR}.")
else:
    print(f"ERROR: Source directory '{DRIVE_SOURCE_DIR}' not found on Google Drive.")
    print("Please check the folder name in your Google Drive and update DRIVE_SOURCE_DIR accordingly.")

Copying TIFF files from /content/drive/MyDrive/PreyLang to /content/tiles...
Successfully copied 16 TIFF files to /content/tiles.


## 1. Sanity checks

In [6]:
info = {}
for t in TILES:
    shapes, masks = {}, {}
    for y in [2019, 2021, 2023, 2025]:
        with rasterio.open(tile_path(y, t)) as src:
            assert src.count == 7 and src.descriptions[6] == "label", (y, t, src.descriptions)
            shapes[y] = (src.height, src.width)
            lab = src.read(7)
        masks[y] = np.isfinite(lab)
        print(f"tile {t} {y}  shape={shapes[y]}  labeled={masks[y].mean():.3f}  "
              f"forest share of labeled={np.nanmean(lab) if masks[y].any() else float('nan'):.3f}")
    assert len(set(shapes.values())) == 1, f"shape differs between years for tile {t}"
    assert all((masks[y] == masks[2019]).all() for y in masks), f"labeled area differs between years for tile {t}"
    info[t] = shapes[2019]
print("OK: same shape and same labeled area in every year")

tile 0_0 2019  shape=(3739, 3273)  labeled=0.567  forest share of labeled=0.157
tile 0_0 2021  shape=(3739, 3273)  labeled=0.567  forest share of labeled=0.085
tile 0_0 2023  shape=(3739, 3273)  labeled=0.567  forest share of labeled=0.028
tile 0_0 2025  shape=(3739, 3273)  labeled=0.567  forest share of labeled=0.000
tile 0_1 2019  shape=(3738, 3273)  labeled=0.174  forest share of labeled=0.264
tile 0_1 2021  shape=(3738, 3273)  labeled=0.174  forest share of labeled=0.152
tile 0_1 2023  shape=(3738, 3273)  labeled=0.174  forest share of labeled=0.054
tile 0_1 2025  shape=(3738, 3273)  labeled=0.174  forest share of labeled=0.000
tile 1_0 2019  shape=(3739, 3272)  labeled=0.165  forest share of labeled=0.505
tile 1_0 2021  shape=(3739, 3272)  labeled=0.165  forest share of labeled=0.342
tile 1_0 2023  shape=(3739, 3272)  labeled=0.165  forest share of labeled=0.112
tile 1_0 2025  shape=(3739, 3272)  labeled=0.165  forest share of labeled=0.000
tile 1_1 2019  shape=(3738, 3272)  label

## 2. Build blocks and assign splits

Blocks with fewer than `MIN_LABELED_PX` labeled pixels are dropped (nothing to learn or test on).
The shuffle uses `SEED`. If the printed table looks unbalanced (for example the test set has almost no 2023 positives), change `SEED` and rerun this cell.

In [7]:
MIN_LABELED_PX = 10_000
blocks = []
for t in TILES:
    H, W = info[t]
    labs = {}
    for y in TRAIN_YEARS:
        with rasterio.open(tile_path(y, t)) as src:
            labs[y] = src.read(7)
    for r0 in range(0, H, BLOCK):
        for c0 in range(0, W, BLOCK):
            h, w = min(BLOCK, H - r0), min(BLOCK, W - c0)
            l = {y: labs[y][r0:r0 + h, c0:c0 + w] for y in TRAIN_YEARS}
            n_lab = int(np.isfinite(l[TRAIN_YEARS[0]]).sum())
            if n_lab < MIN_LABELED_PX:
                continue
            rec = dict(id=f"{t}_{r0}_{c0}", tile=t, r0=r0, c0=c0, h=h, w=w, n_labeled=n_lab)
            for y in TRAIN_YEARS:
                rec[f"pos_{y}"] = int(np.nansum(l[y]))
            blocks.append(rec)
    del labs

rng = np.random.default_rng(SEED)
order = rng.permutation(len(blocks))
total = sum(b["n_labeled"] for b in blocks)
cum = 0
for i in order:
    b = blocks[i]
    f = cum / total
    b["split"] = "train" if f < 0.70 else ("val" if f < 0.85 else "test")
    cum += b["n_labeled"]

df = pd.DataFrame(blocks)
summary = df.groupby("split").agg(blocks=("id", "count"), labeled_px=("n_labeled", "sum"),
                                  **{f"pos_{y}": (f"pos_{y}", "sum") for y in TRAIN_YEARS})
for y in TRAIN_YEARS:
    summary[f"pos_rate_{y}"] = (summary[f"pos_{y}"] / summary["labeled_px"]).round(3)
print(summary)
for s in ["train", "val", "test"]:
    assert s in summary.index, f"split {s} is empty - lower MIN_LABELED_PX or change SEED"
    if summary.loc[s, f"pos_{TRAIN_YEARS[-1]}"] < 5000:
        print(f"WARNING: {s} has few {TRAIN_YEARS[-1]} positives - consider another SEED")

       blocks  labeled_px  pos_2019  pos_2021  pos_2023  pos_rate_2019  \
split                                                                    
test        4     1753356    295426    186846     71766          0.168   
train      38     9945616   2770907   1592159    566337          0.279   
val         7     2047354    515272    319515     85022          0.252   

       pos_rate_2021  pos_rate_2023  
split                                
test           0.107          0.041  
train          0.160          0.057  
val            0.156          0.042  


## 3. Normalization stats (train blocks only) and save

In [8]:
rng = np.random.default_rng(SEED)
samples = []
for b in [b for b in blocks if b["split"] == "train"]:
    for y in TRAIN_YEARS:
        a = read_window(y, b["tile"], b["r0"], b["c0"], b["h"], b["w"])
        bands = a[:6].reshape(6, -1)
        idx = np.flatnonzero(np.isfinite(bands).all(0))
        if len(idx) == 0:
            continue
        sel = rng.choice(idx, min(5000, len(idx)), replace=False)
        samples.append(bands[:, sel])
S = np.concatenate(samples, 1)
norm = {"bands": BANDS, "mean": S.mean(1).tolist(), "std": S.std(1).tolist()}
print(pd.DataFrame({"mean": norm["mean"], "std": norm["std"]}, index=BANDS).round(4))

json.dump({"block": BLOCK, "seed": SEED, "train_years": TRAIN_YEARS, "blocks": blocks},
          open(f"{WORK_DIR}/splits.json", "w"), indent=1)
json.dump(norm, open(f"{WORK_DIR}/norm_stats.json", "w"), indent=1)
df.to_csv(f"{WORK_DIR}/block_stats.csv", index=False)
print("saved to", WORK_DIR)

       mean     std
B2   0.0445  0.0162
B3   0.0641  0.0192
B4   0.0585  0.0339
B8   0.2646  0.0403
B11  0.1968  0.0615
B12  0.1100  0.0570
saved to /content/drive/MyDrive/PreyLang_work
